In [1]:
!nvidia-smi

Wed Mar 11 06:27:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers
!pip install datasets
!pip install peft
!pip install accelerate
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.8 MB/s eta 0:00:00


In [3]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset

dataset = load_dataset("rootsautomation/RICO-Screen2Words")
dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/test-00001-of-00002.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15743 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/2364 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4310 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'image_semantic'],
        num_rows: 15743
    })
    val: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'image_semantic'],
        num_rows: 2364
    })
    test: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'imag

In [5]:
dataset["train"][0]

{'screenId': 2,
 'captions': ['notification displaying a calendar with done option',
  'pop up displaying a calendar',
  'pop up displaying calendar to select date',
  'pop up window of calendar with options',
  'pop-up of a calendar to select date of birth'],
 'file_name': 'rico/2.jpg',
 'app_package_name': 'yong.app.videoeditor',
 'play_store_name': 'Video Maker Pro Free',
 'category': 'Video Players & Editors',
 'average_rating': 3.4,
 'number_of_ratings': '100637',
 'number_of_downloads': '  5,000,000 - 10,000,000  ',
 'file_name_icon': 'rico_icons/yong.app.videoeditor.png',
 'file_name_semantic': 'rico_semantic_annotations/2.png',
 'semantic_annotations': '{"ancestors": ["android.widget.FrameLayout", "android.view.ViewGroup", "android.view.View", "java.lang.Object"], "class": "com.android.internal.policy.PhoneWindow$DecorView", "bounds": [92, 380, 1348, 2095], "clickable": false, "children": [{"ancestors": ["android.view.ViewGroup", "android.view.View", "java.lang.Object"], "class

In [6]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [8]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
)

model = get_peft_model(model, config)

In [10]:
def preprocess(example):
    caption = example["captions"][0]   # take the first caption

    inputs = processor(
        images=example["image"],
        text=caption,
        return_tensors="pt"
    )

    inputs["labels"] = inputs["input_ids"]
    return inputs

In [16]:
def preprocess(example):
    caption = example["captions"][0]

    inputs = processor(
        images=example["image"],
        text=caption,
        padding="max_length",
        truncation=True,
        max_length=50,
        return_tensors="pt"
    )

    inputs["labels"] = inputs["input_ids"]

    return {k: v.squeeze() for k, v in inputs.items()}


dataset = dataset.select(range(500))

dataset = dataset.map(preprocess)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-5,
)

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [18]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=125, training_loss=11.5996123046875, metrics={'train_runtime': 183.7828, 'train_samples_per_second': 2.721, 'train_steps_per_second': 0.68, 'total_flos': 2.983178502144e+17, 'train_loss': 11.5996123046875, 'epoch': 1.0})

In [19]:
model.push_to_hub("rico-blip-lora-model")
processor.push_to_hub("rico-blip-lora-model")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  12%|#1        |  550kB / 4.73MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/suhas2468/rico-blip-lora-model/commit/c1337deff59f5b48b6bde3b025d234e6ea3371a4', commit_message='Upload processor', commit_description='', oid='c1337deff59f5b48b6bde3b025d234e6ea3371a4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/suhas2468/rico-blip-lora-model', endpoint='https://huggingface.co', repo_type='model', repo_id='suhas2468/rico-blip-lora-model'), pr_revision=None, pr_num=None)

In [21]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png"

image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="pt").to("cuda")

out = model.generate(**inputs)

processor.decode(out[0], skip_special_tokens=True)

'a cat laying on a pile of yarn'